#### 每个comment都有对应的post

In [2]:
import pandas as pd
root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
# 文件路径设置（根据实际情况修改）
comments_file = root + 'comments.csv'  # 评论文件
posts_file = root + 'posts.csv'        # 帖子文件

# 读取文件
comments_df = pd.read_csv(comments_file)
posts_df = pd.read_csv(posts_file)

# 获取 posts 文件中所有帖子 id:ID 列的值（即有效的 post id 列表）
valid_post_ids = posts_df['id:ID'].tolist()

# 过滤出 comment 文件中，post 属性在 valid_post_ids 中的行
filtered_comments_df = comments_df[comments_df['post'].isin(valid_post_ids)]

save_root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/valid_data/'
# 保存过滤后的数据到新的 CSV 文件
filtered_comments_df.to_csv(save_root + 'comments_valid.csv', index=False)

print(f"已删除 {len(comments_df) - len(filtered_comments_df)} 个无对应帖子的评论，共剩余 {len(filtered_comments_df)} 个评论。")


已删除 820 个无对应帖子的评论，共剩余 33525 个评论。


#### 每个user都有对应的comment和post

In [ ]:
import pandas as pd

root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/valid_data/'

# 定义文件路径（根据实际情况修改）
users_file = root + 'users.csv'
posts_file = root + 'posts_valid.csv'
comments_file = root + 'comments_valid.csv'

# 读取文件
users_df = pd.read_csv(users_file)
posts_df = pd.read_csv(posts_file)
comments_df = pd.read_csv(comments_file)

# 从 posts 和 comments 文件中分别提取 creator 属性的用户 ID
posts_creators = set(posts_df['creator'].unique())
comments_creators = set(comments_df['creator'].unique())

# 合并两个集合，得到所有在 posts 或 comments 中出现过的用户 ID
valid_user_ids = posts_creators.union(comments_creators)

# 筛选出 users 文件中 id:ID 在 valid_user_ids 内的元组
filtered_users_df = users_df[users_df['id:ID'].isin(valid_user_ids)]

# 保存过滤后的用户数据到新文件
output_file = root + 'users_valid.csv'
filtered_users_df.to_csv(output_file, index=False)

print(f"已删除 {len(users_df) - len(filtered_users_df)} 个无对应 posts 或 comments 的用户，共保留 {len(filtered_users_df)} 个用户，结果保存为 '{output_file}'")


#### 分别统计有多少个post和comment找不到对应的user

In [5]:
import pandas as pd

# root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/valid_data/'
root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'

# 读取文件
users_df = pd.read_csv(root + 'users_valid_test2.csv')
posts_df = pd.read_csv(root + 'posts.csv')
comments_df = pd.read_csv(root + 'comments.csv')

# 提取 users 文件中的有效用户 ID
valid_user_ids = set(users_df['id:ID'])

# 找出 posts 中 creator 不在有效用户中的记录
posts_missing = posts_df[~posts_df['creator'].isin(valid_user_ids)]
# 找出 comments 中 creator 不在有效用户中的记录
comments_missing = comments_df[~comments_df['creator'].isin(valid_user_ids)]

# 只保留需要的属性： "id:ID" 和 "creator"
posts_missing = posts_missing[['id:ID', 'creator']]
comments_missing = comments_missing[['id:ID', 'creator']]

# 添加一个标识列，方便区分来源
posts_missing['datatype'] = 'post'
comments_missing['datatype'] = 'comment'

# # 合并 posts 和 comments 中找不到对应用户的记录
# lost_users_df = pd.concat([posts_missing, comments_missing], ignore_index=True)
# 
# # 保存到 pc_lost_user.csv 文件中
# lost_users_df.to_csv(root + 'pc_lost_user.csv', index=False)
# 
# print("已将找不到对应用户的 post 和 comment 的 id:ID 与 creator 保存到 'pc_lost_user.csv' 文件中。")

print(f"找不到对应用户的 post 数量：{len(posts_missing)}")
print(f"找不到对应用户的 comment 数量：{len(comments_missing)}")


找不到对应用户的 post 数量：245
找不到对应用户的 comment 数量：8799


#### 我所有用户的信息，保存在了dataset文件夹下，下面有许多user.csv文件, 查找

In [ ]:
import pandas as pd
import glob
import os
from tqdm import tqdm

root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/valid_data/'
# 定义文件路径
pc_lost_user_file = root + 'pc_lost_user.csv'
dataset_folder = '/home/wangshuo/resource/datasets/parler_data/valid_users/'  # 假设所有用户文件保存在这个文件夹下

# 读取 pc_lost_user.csv，获取所有缺失用户的 creator 值
pc_lost_df = pd.read_csv(pc_lost_user_file)
missing_creators = set(pc_lost_df['creator'].unique())

# 获取 dataset 文件夹下所有 CSV 文件（假设这些文件均为用户文件）
user_files = glob.glob(os.path.join(dataset_folder, '*.csv'))

# 初始化列表，存放所有查找到的用户记录
lost_user_rows = []

# 遍历每个用户文件，使用 tqdm 显示进度条
for file in tqdm(user_files, desc="Processing user files", unit="file"):
    # 可选：判断文件名是否符合用户文件命名规则，例如包含 "user" 关键字
    if 'user' not in os.path.basename(file).lower():
        continue
    user_df = pd.read_csv(file)
    # 假设用户文件中的用户 ID 字段名称为 "id:ID"
    filtered = user_df[user_df['id'].isin(missing_creators)]
    if not filtered.empty:
        lost_user_rows.append(filtered)

# 合并所有查找到的用户记录，并保存到 user_lost.csv
if lost_user_rows:
    lost_users_df = pd.concat(lost_user_rows, ignore_index=True)
    lost_users_df.to_csv(root + 'user_lost.csv', index=False)
    print(f"已找到 {len(lost_users_df)} 个丢失的用户，保存到 'user_lost.csv'。")
else:
    print("未找到丢失的用户。")


#### id:ID,badges,banned,bio,posts,comments,user_followers,user_following,username,ucNum,upNum
user_lost文件中只保留上面属性，并将更新后的user_lost文件的所有元组，添加到users_valid.csv文件中

In [10]:
import pandas as pd

# 定义文件路径（根据实际情况修改）
root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/valid_data/'
user_lost_file = root + 'user_lost.csv'
users_valid_file = root + 'users_valid.csv'

# 读取 user_lost.csv 文件
user_lost_df = pd.read_csv(user_lost_file)

# 只保留指定的属性
columns_to_keep = ['id:ID', 'badges', 'banned', 'bio', 'posts', 'comments', 'user_followers', 'user_following', 'username']
user_lost_df = user_lost_df[columns_to_keep]

# 读取 users_valid.csv 文件
users_valid_df = pd.read_csv(users_valid_file)

# 将更新后的 user_lost 数据追加到 users_valid 数据中
combined_df = pd.concat([users_valid_df, user_lost_df], ignore_index=True)

# 保存合并后的数据到 users_valid.csv 文件中
combined_df.to_csv(users_valid_file, index=False)

print(f"已将更新后的 user_lost 数据添加到 '{users_valid_file}' 文件中。")


已将更新后的 user_lost 数据添加到 '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/valid_data/users_valid.csv' 文件中。


#### 每个comment和post都有对应的user